# NAMformer: Transformer-enhanced additive model

NAMformer contextualizes all feature tokens, adds a global CLS-token head, and retains token-level and optional explicit interaction contributions.


## Model


$$
h=\operatorname{Transformer}(E(x)),\qquad
\eta(x)=g(h_{\mathrm{CLS}})+\sum_jf_j(e_j)
+\sum_{S\in\mathcal I}f_S(e_S)+\beta_0.
$$

The global CLS term means the model is decomposed but not purely univariate-additive.


## Shared estimator API

All neural estimators use `fit`, `predict`, `score`, `evaluate`, and
`predict_components`. The component result reconstructs predictions on the link
scale and supports shared term-importance and plotting utilities. Constructor
options such as `numerical_preprocessing` and `categorical_preprocessing` are forwarded to
PreTab and are fitted on training rows only.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(7)
n = 180
X = pd.DataFrame({
    "x1": rng.uniform(-1.0, 1.0, n),
    "x2": rng.normal(size=n),
    "group": rng.choice(["a", "b", "c"], size=n),
})
y = (
    np.sin(np.pi * X["x1"])
    + 0.35 * X["x2"] ** 2
    + 0.30 * (X["group"] == "b")
    + rng.normal(0.0, 0.12, n)
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

# Set True to run the small fit and all fitted-model demonstrations.
RUN_TRAINING = False


## Construct the estimator


In [ ]:
from nampy.models import NAMformerClassifier, NAMformerLSS, NAMformerRegressor


model = NAMformerRegressor(
    d_model=16,
    n_layers=2,
    n_heads=2,
    transformer_dim_feedforward=32,
    head_layer_sizes=(16,),
    interactions=(("x1", "x2"),),
    attn_dropout=0.0,
)
model.get_params(deep=False)


## Fit and inspect

Enable `RUN_TRAINING` above for a short demonstration. Real work should use a
larger validation set, enough epochs, and early stopping.


In [ ]:
if RUN_TRAINING:
    model.fit(
        X_train,
        y_train,
        max_epochs=3,
        batch_size=64,
        random_state=7,
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    predictions = model.predict(X_test)
    r2 = model.score(X_test, y_test)
    metrics = model.evaluate(X_test, y_test)
    components = model.predict_components(X_test, center=True)
    components.validate_additive_reconstruction()
    display({"R2": r2, **metrics})
    display(model.term_importance(X_test).head())


## Model-specific controls

The CLS head captures unrestricted global context. Token heads and explicitly configured interaction heads remain available through `predict_components`.


In [ ]:
if RUN_TRAINING:
    components = model.predict_components(X_test)
    display(components.terms.keys())
    display(model.interaction_importance(X_test))


## Task variants and limits

NAMformer provides regression, classification, and LSS estimators.
